In [1]:
import sys 
import os

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
plt.style.use('ggplot')

In [3]:
from utils.plots import plot_numerical_data

# Read Dataset

In [4]:
pipeline_name = "Pipeline3"

In [9]:
X_train = pd.read_parquet(fr'..\data\feature_eng\X_train_feat_eng_{pipeline_name}.parquet')
y_train = pd.read_parquet(fr'..\data\feature_eng\y_train_feat_eng_{pipeline_name}.parquet')

In [10]:
df = pd.concat([X_train, y_train], axis=1)

In [11]:
print(f'Dataset rows and columns: {df.shape}')

Dataset rows and columns: (712, 13)


In [12]:
df.head()

,numerical_pipe_con__Age,numerical_pipe_con__Fare,numerical_pipe_dis__SibSp,numerical_pipe_dis__Parch,numerical_pipe_dis__IsAlone,numerical_pipe_dis__FamilySize,categorical_pipe__Pclass,categorical_pipe__Sex,categorical_pipe__Ticket,categorical_pipe__Embarked,categorical_pipe__Title,categorical_pipe__Cabin_1p,Survived
331,0.356663,0.350768,-0.130931,-0.479342,0.812203,-0.182707,1.614136,-0.724310,0.0,-0.566021,-0.791063,1.303385,0
733,0.356663,-0.707482,-0.130931,-0.479342,0.812203,-0.182707,0.400551,-0.724310,0.0,-0.566021,-0.791063,-0.482394,0
382,0.356663,-0.707482,-0.130931,-0.479342,0.812203,-0.182707,-0.813034,-0.724310,0.0,-0.566021,-0.791063,-0.482394,0
704,0.356663,-0.707482,-0.130931,-0.479342,-1.231219,-0.182707,-0.813034,-0.724310,0.0,-0.566021,-0.791063,-0.482394,0
813,-2.920036,0.350768,-0.130931,2.048742,-1.231219,5.473255,-0.813034,1.380624,0.0,-0.566021,1.067805,-0.482394,0


In [ ]:
num_var = df.select_dtypes(include=['number']).columns
num_var

# Exploratory Analisys

In [ ]:
df.info()

# Check NA values

In [ ]:
df.isna().mean().plot.bar(figsize=(8,4))

# Check Cardinality

In [ ]:
for col in num_var:
    print('Numero de labels por variável na coluna: ' + col + ' ' + str(df[col].nunique()))

# Plots

In [ ]:
plot_numerical_data(df, target='Survived', classification=True)

# Correlation Analysis

In [ ]:
from feature_engine.selection import DropCorrelatedFeatures, SmartCorrelatedSelection

In [ ]:
sel = DropCorrelatedFeatures(
    threshold=0.8,
    method='pearson',
    missing_values='ignore'
)


# find correlated features

sel.fit(df)

In [ ]:
sel.correlated_feature_sets_

## SmartCorrelationSelection

### Model Performance

We will keep a feature from each correlation group based on the performance of a random forest.

In [ ]:
# random forest
rf = RandomForestClassifier(
    n_estimators=10,
    random_state=20,
)

# correlation selector
sel = SmartCorrelatedSelection(
    variables=None, # if none, selector examines all numerical variables
    method="pearson",
    threshold=0.8,
    missing_values="raise",
    selection_method="model_performance",
    estimator=rf,
    scoring="roc_auc",
    cv=3,
)

sel.fit(X_train, y_train.values.ravel())

In [ ]:
sel.correlated_feature_sets_

In [ ]:
sel.features_to_drop_